In [1]:
import geopandas as gpd
import os
import pandas as pd


#=================
#Data loading
#=================

#Change base path for your own file system
GTFS_data = {}
base_path = r"C:\Users\luca\Downloads\Lux_public_transport_project\gtfs-20260422-20260508"

for file in os.listdir(base_path):
    if file.endswith(".txt"):
        name = file.replace(".txt", "")
        full_path = os.path.join(base_path, file)
        GTFS_data[name] = pd.read_csv(full_path)

#Normalize IDs
id_columns = {
    "stops": ["stop_id", "parent_station"],
    "stop_times": ["stop_id", "trip_id"],
    "trips": ["service_id", "trip_id", "route_id"],
    "calendar": ["service_id"],
    "routes": ["route_id", "agency_id"]
}

for table_name, cols in id_columns.items():
    df = GTFS_data[table_name]
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()


#===============
#Data validation
#===============

stops = GTFS_data["stops"]
stop_times = GTFS_data["stop_times"]
trips = GTFS_data["trips"]
calendar = GTFS_data["calendar"]
routes = GTFS_data["routes"]

#Checks the data relation: calendar -> trips -> stop_times -> stops
logic_checks = [
    trips["service_id"].isin(calendar["service_id"]).all(),
    stop_times["trip_id"].isin(trips["trip_id"]).all(),
    stop_times["stop_id"].isin(stops["stop_id"]).all()
]

if not all(logic_checks):
    raise ValueError("GTFS ID relationship check failed")

#Checks if stop and trip IDs are unique
unique_checks = [
    stops["stop_id"].is_unique,
    trips["trip_id"].is_unique
]

if not all(unique_checks):
    raise ValueError("ID uniqueness check failed")

#Checks if stop sequence or departure time is reversed.
chrono = stop_times.sort_values(["trip_id", "stop_sequence"])
chrono["departure_time"] = pd.to_timedelta(chrono["departure_time"])
chrono["stop_sequence"] = pd.to_numeric(chrono["stop_sequence"])

if (
    (chrono.groupby("trip_id")["stop_sequence"].diff() < 0) |
    (chrono.groupby("trip_id")["departure_time"].diff() < pd.Timedelta(0))
).any():
    raise ValueError("Stop order check failed")

#==============
#Data cleaning
#==============

#Removes null and 0 values in stops
stops = stops.dropna(subset=["stop_lat", "stop_lon"])
stops = stops[(stops["stop_lat"] != 0) & (stops["stop_lon"] != 0)]

#Converts times into seconds to deal with 24 hours clock bugs
stop_times["dep_sec"] = pd.to_timedelta(stop_times["departure_time"]).dt.total_seconds()
stop_times["arr_sec"] = pd.to_timedelta(stop_times["arrival_time"]).dt.total_seconds()

#Removes any duplicate trips. Useful when forming PT edges
stop_times = stop_times.drop_duplicates(subset=["trip_id", "stop_sequence"])

#===========
#Find valid trips, AVL + weekday service {111:tram, 6:bus}
#===========
valid_agency_ids = ["6", "111"]

valid_route_ids = set(routes[routes["agency_id"].isin(valid_agency_ids)]["route_id"].astype(str))

weekdays = ["monday", "tuesday", "wednesday", "thursday", "friday"]
weekday_service_ids = set(calendar.loc[calendar[weekdays].eq(1).all(axis=1), "service_id"])

valid_trip_ids = set(trips[trips["route_id"].isin(valid_route_ids) & trips["service_id"].isin(weekday_service_ids)]["trip_id"])

#===========
#Filter to peak hours, 7 am to 9 am
#===========
peak_start = 3600 * 7
peak_end   = 3600 * 9

peak_times = stop_times[
    stop_times["dep_sec"].between(peak_start, peak_end) &
    stop_times["trip_id"].isin(valid_trip_ids)
]

#===========
#We only consider AVL stops
#===========
avl_stop_ids = set(peak_times["stop_id"])
stops = stops[stops["stop_id"].isin(avl_stop_ids)]

#==============
#Relational event table
#==============

events = peak_times.merge(trips, on="trip_id", how="inner", validate="many_to_one")
events = events.merge(stops, on="stop_id", how="inner", validate="many_to_one")
events_unique = events.sort_values(["trip_id", "stop_sequence"]).drop_duplicates(subset=["trip_id", "stop_id"], keep="first")

#========================
#Find Line Level Headway data
#========================

#We group the events by stop and route ID so we look at the headway of every line at every stop.
#What this means is that at a single stop, if say line 3 and line 5 use that stop, we consider their two avg headways separately.
#This lets us find the fastest route based on the best line, a realistic approximation.
freq = events_unique.groupby(["stop_id", "route_id"])["dep_sec"].count().rename("n_departures").reset_index()

PEAK_WINDOW_MIN = 120
freq["avg_headway"] = PEAK_WINDOW_MIN / freq["n_departures"]

#========================
#Output Dataset Production
#========================

#Keep stops and headway data separate. Headway loaded as CSV when forming PT edges.
stops_out = stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

stops_gdf = gpd.GeoDataFrame(
    stops_out,
    geometry=gpd.points_from_xy(stops_out.stop_lon, stops_out.stop_lat)
).set_crs(epsg=4326)

stop_route_headways = freq[["stop_id", "route_id", "n_departures", "avg_headway"]]

#========================
#Save outputs
#========================

stops_gdf.to_file(
    r"C:\Users\luca\Downloads\Lux_public_transport_project\stop_freq_avl.gpkg",
    driver="GPKG"
)
stop_route_headways.to_csv("stop_route_headways.csv", index=False)
trips.to_csv("trips.csv", index=False)
peak_times.to_csv("stop_times.csv", index=False)

print("GTFS done")


GTFS done
